# Excel reports through ZEMI Arsenal and Guidance

Arsenal loads and starts Qwen 3.5 4B, `MarkItDown` converts Excel to compact Markdown, and Guidance constrains generation with the `Reports` Pydantic schema. This example does not use `zemi.exp`, direct TOML reading, or external GBNF.

## Preparing and starting Arsenal

In [ ]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession


arsenal = ArsenalSession("@comp/playbook.toml")
# arsenal.download()  # Uncomment to download all resources in advance.
zemi.arsenal.begin(arsenal, 
    stop_before_begin=True,
    llama_router_mode=False,
)

assistant_config = (
    arsenal.llamas["primary"]
    .models["qwen"]
    .assistants["report_parser"]
)
model = assistant_config.clients.guidance.model
model.echo = False

## Compact result schema

Guidance accepts a Pydantic class directly in `guidance.json`. Its internal mechanism compiles the schema and constrains allowed tokens during generation.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Transaction(StrictModel):
    date: str = Field(description="Date in YYYY-MM-DD format")
    article: str
    cost: float


class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description="Date in YYYY-MM-DD format")
    manager: str
    transactions: list[Transaction]


class Reports(StrictModel):
    reports: list[Report]

## Excel → Markdown

In [ ]:
from IPython.display import Markdown, display
from markitdown import MarkItDown

from zemi import env


data_dir = env.path.comp.root / "data/case01"
excel_files = [data_dir / f"Report {number}.xlsx" for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)

excel_context = "\n\n".join(
    f"# File: {path.name}\n\n{converter.convert(path).text_content.strip()}"
    for path in excel_files
)

print("Prepared files:")
for path in excel_files:
    print(f"- {path.name}: {path.stat().st_size} bytes")
print(f"\nMarkdown context size: {len(excel_context)} characters")
display(Markdown(excel_context))

## Structured extraction with Guidance

In [ ]:
import json

from guidance import assistant, json as gen_json, system, user


task = """
Process all provided Excel reports. For each file, extract the file name,
city/branch, export date, manager, and table rows (date, article, cost).
Do not include the “Total” row or invent missing data.
Write all dates strictly in YYYY-MM-DD format.
""".strip()

lm = model
with system():
    lm += (
        "You carefully convert Excel reports into strictly "
        "structured data without inventing anything."
    )
with user():
    lm += f"{task}\n\n{excel_context}"
with assistant():
    lm += gen_json(
        name="reports_json",
        schema=Reports,
        temperature=0.0,
        max_tokens=2048,
    )

result = Reports.model_validate_json(lm["reports_json"])
print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))

In [ ]:
result

## Stopping Arsenal

Run this cell when the model is no longer needed.

In [ ]:
zemi.arsenal.end(arsenal, stop_after_end=True)